# MASLD Public Atlas — Cluster 3 vs 4 Analysis

**목적:** MASLD monocyte 서브클러스터링에서 SLC7A5 high cluster의 특징을 구하기 위한 분석

---
## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import mannwhitneyu, ttest_ind
import os

try:
    import gseapy as gp
    GSEAPY_AVAILABLE = True
    print(f"gseapy version: {gp.__version__}")
except ImportError:
    GSEAPY_AVAILABLE = False
    print("gseapy not found — GSEA section will be skipped. Install with: pip install gseapy")

# PDF 저장 시 폰트 깨짐 방지
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')

# 저장 디렉토리
FIG_DIR = "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/"
os.makedirs(FIG_DIR, exist_ok=True)

print("Setup complete.")

---
## 2. Data Loading

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")
adata

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono_cellannot.h5ad")
adata

In [ ]:
# raw count 데이터 설정 (normalize → log1p → .raw에 저장)
if adata.raw is None:
    print("raw data not exists")
else:
    print("raw data already exists")

In [ ]:
# 클러스터 세포 수 확인
print(adata.obs['celltype'].value_counts().sort_index())

In [ ]:
# DEG for all cells
sc.tl.rank_genes_groups(
    adata,
    groupby='celltype',
    method='wilcoxon',
    use_raw=True,
    pts=True
)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

In [ ]:
deg_all = sc.get.rank_genes_groups_df(adata, group=None)
deg_all

In [ ]:
deg_all.to_csv("/home/jaehyunchoi//Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/MASLD_cellseq_DEG_all_clusters.csv", index=False)

---
## 3. Overview Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.umap(adata, color='10_cluster', frameon=False,
           title='mono/MAC clusters (leiden_9)', size=10, ax=axes[0], show=False)

sc.pl.umap(adata, color='state', frameon=False,
           title='Condition (MASLD / Control)', size=10, ax=axes[1], show=False)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/cluster3v4_overview_umap.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# 핵심 마커 유전자 Feature plot — Treg vs PD1high 비교
key_markers = ['CD14', 'FCGR3A', 'CCR2', 'CX3CR1', 'SELL', 'LYZ','VCAN']

sc.pl.umap(
    adata,
    color=key_markers,
    cmap='Reds',
    frameon=False,
    size=10,
    ncols=3,
    show=False
)
#plt.savefig(f"{FIG_DIR}/cluster3v4_key_markers_featureplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['CD14', 'FCGR3A', 'CCR2', 'CX3CR1', 'SELL', 'LYZ','VCAN']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(8.5, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        use_raw=False,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

---
## 4. Gene Set Definition

| Gene Set |
|----------|
| **monocyte** |
| **MAC** |
| **DC** |
| **Kupffer** | 

In [ ]:
# ── Gene Set 정의 ─────────────────────────────────────────────────────────

mono_genes = ['CD14','FCGR3A','CCR2','CX3CR1','SELL','LYZ','S100A8','S100A9','VCAN','GNA15','ITGAM'
]

mac_genes = ['CD68','MRC1','MSR1','TREM2','APOE','C1QA','C1QB','C1QC','LYVE1','FCGR1A','SPP1']

dc_genes = ['ITGAX','FLT3','CLEC9A','XCR1','CLEC1A','SIGLEC6','LILRA4','IRF8','CCR7','HLA-DRA']

kupf_genes = ['VSIG4','CLEC4F','TIMD4','CD163','MARCO','SLC40A1','ID3','FOLR2','KLF4','HMOX1']

gene_sets = {
    'Mono':    mono_genes,
    'MAC':     mac_genes,
    'DC':     dc_genes,
    'Kupffer': kupf_genes,
}

# 실제 데이터에 존재하는 유전자만 필터링
all_genes = set(adata.raw.var_names)
gene_sets_filtered = {}
for gs_name, gs_list in gene_sets.items():
    filtered = [g for g in gs_list if g in all_genes]
    missing = [g for g in gs_list if g not in all_genes]
    gene_sets_filtered[gs_name] = filtered
    print(f"{gs_name}: {len(filtered)}/{len(gs_list)} genes found", end="")
    if missing:
        print(f" | missing: {missing}")
    else:
        print()

print("\nGene sets defined.")

---
## 5. Gene Set Scoring per Cluster (sc.tl.score_genes)

In [ ]:
# 각 gene set 점수 계산
for gs_name, gs_list in gene_sets_filtered.items():
    if len(gs_list) < 3:
        print(f"Skipping {gs_name}: too few genes ({len(gs_list)})")
        continue
    sc.tl.score_genes(adata, gene_list=gs_list, score_name=f"{gs_name}_score")
    print(f"{gs_name}_score computed.")

print("\nAll gene set scores computed.")

In [ ]:
# Gene set score UMAP
score_cols = [f"{k}_score" for k in gene_sets_filtered.keys()]

sc.pl.umap(
    adata,
    color=score_cols,
    cmap='RdBu_r',
    vcenter=0,
    frameon=False,
    size=10,
    ncols=2,
    show=False
)
plt.savefig(f"{FIG_DIR}/cluster3v4_geneset_scores_umap.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Gene set score — cluster별 violin plot
#palette = adata_CD4.uns['leiden_9_colors']
n_scores = len(score_cols)

fig, axes = plt.subplots(n_scores, 1, figsize=(8, 3 * n_scores))
if n_scores == 1:
    axes = [axes]

for ax, col in zip(axes, score_cols):
    sc.pl.violin(
        adata,
        keys=col,
        groupby='10_cluster',
        #palette=palette,
        ax=ax,
        show=False,
        stripplot=False,
        jitter=False
    )
    ax.set_title(col.replace('_score', '') + ' Gene Set Score')
    ax.axvline(x=3.5, color='gray', linestyle='--', linewidth=0.8)  # cluster 3 경계
    ax.axvline(x=4.5, color='red', linestyle='--', linewidth=0.8, label='cluster 4')  # cluster 4 경계

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/cluster3v4_geneset_scores_violin.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Gene set score — cluster별 dotplot (cluster 3, 4 강조)
sc.pl.dotplot(
    adata,
    var_names=score_cols,
    groupby='10_cluster',
    standard_scale='var',
    show=False
)
plt.savefig(f"{FIG_DIR}/cluster3v4_geneset_scores_dotplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

---
## 6. Metabolic Pathway GSEA — All Clusters

- 비교: **각 cluster vs Rest** (Wilcoxon preranked)
- Pathways: **Glycolysis / Gluconeogenesis / Oxidative Phosphorylation / TCA Cycle**
- 시각화: NES heatmap · Bubble plot · ES curve overlay


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6a. Metabolic Pathway Gene Sets
#     출처: KEGG hsa00010 / hsa00020 / hsa00190 (curated)
# ══════════════════════════════════════════════════════════════════════════

glycolysis_genes = [
    'GCK','HK1','HK2','HK3',
    'PFKL','PFKM','PFKP',
    'ALDOA','ALDOB','ALDOC',
    'TPI1',
    'GAPDH',
    'PGK1','PGK2',
    'PGAM1','PGAM2','PGAM4','BPGM',
    'ENO1','ENO2','ENO3',
    'PKM','PKLR',
    'LDHA','LDHB','LDHC','LDHD',
]

gluconeogenesis_genes = [
    # gluconeogenesis-specific enzymes
    'PC',                          # pyruvate carboxylase
    'PCK1','PCK2',                 # PEPCK (cytosolic / mito)
    'FBP1','FBP2',                 # fructose-1,6-bisphosphatase
    'G6PC','G6PC2','G6PC3',        # glucose-6-phosphatase
    # shared reversible steps
    'ENO1','ENO2','ENO3',
    'PGAM1','PGAM2',
    'PGK1','PGK2',
    'GAPDH',
    'TPI1',
    'ALDOA','ALDOB','ALDOC',
    'PFKL','PFKM','PFKP',
    # anaplerotic / auxiliary
    'MDH1','MDH2',
    'GOT1','GOT2',
    'AGXT','AGXT2',
    'SLC25A11','SLC25A13',
]

oxphos_genes = [
    # Complex I — NADH:ubiquinone oxidoreductase
    'NDUFA1','NDUFA2','NDUFA3','NDUFA4','NDUFA5','NDUFA6','NDUFA7',
    'NDUFA8','NDUFA9','NDUFA10','NDUFA11','NDUFA12','NDUFA13',
    'NDUFB1','NDUFB2','NDUFB3','NDUFB4','NDUFB5','NDUFB6','NDUFB7',
    'NDUFB8','NDUFB9','NDUFB10','NDUFB11',
    'NDUFC1','NDUFC2',
    'NDUFS1','NDUFS2','NDUFS3','NDUFS4','NDUFS5','NDUFS6','NDUFS7','NDUFS8',
    'NDUFV1','NDUFV2','NDUFV3',
    # Complex II — succinate:ubiquinone oxidoreductase
    'SDHA','SDHB','SDHC','SDHD',
    # Complex III — cytochrome bc1
    'UQCRC1','UQCRC2','UQCRB','UQCRQ','UQCRFS1','CYC1','UQCR10','UQCR11',
    # Complex IV — cytochrome c oxidase
    'COX4I1','COX4I2','COX5A','COX5B',
    'COX6A1','COX6A2','COX6B1','COX6B2','COX6C',
    'COX7A1','COX7A2','COX7B','COX7C','COX8A','COX8C',
    # Complex V — ATP synthase
    'ATP5F1A','ATP5F1B','ATP5F1C','ATP5F1D','ATP5F1E',
    'ATP5PB','ATP5PD','ATP5PF','ATP5PO','ATP5MC1','ATP5MC2','ATP5MC3',
    'ATP5MF','ATP5MG','ATP5MJ','ATP5MK',
    # Electron carrier
    'CYCS',
]

tca_genes = [
    # Pyruvate dehydrogenase complex
    'PDHA1','PDHA2','PDHB','DLAT','DLD','PDHX',
    # TCA enzymes
    'CS',                          # citrate synthase
    'ACO1','ACO2',                 # aconitase
    'IDH1','IDH2',                 # isocitrate dehydrogenase (NADP+)
    'IDH3A','IDH3B','IDH3G',       # isocitrate dehydrogenase (NAD+)
    'OGDH','OGDHL','DLST','DLD',   # alpha-ketoglutarate dehydrogenase
    'SUCLA2','SUCLG1','SUCLG2',    # succinyl-CoA synthetase
    'SDHA','SDHB','SDHC','SDHD',   # succinate dehydrogenase (TCA + OXPHOS CII)
    'FH',                          # fumarate hydratase
    'MDH1','MDH2',                 # malate dehydrogenase
    # Anaplerosis
    'PC','ME1','ME2','ME3',
    'ACLY',
]

fao_genes = [
    'CPT1A','CPT1B','CPT2',        # carnitine transport
    'ACADVL','ACADL','ACADM','ACADS',  # acyl-CoA dehydrogenases
    'HADHA','HADHB',               # mitochondrial trifunctional protein
    'ACAA2',                       # thiolase
]

metab_sets = {
    'Glycolysis':       glycolysis_genes,
    'Gluconeogenesis':  gluconeogenesis_genes,
    'OxPhos':           oxphos_genes,
    'TCA_Cycle':        tca_genes,
    'FAO':        fao_genes,
}

# 데이터에 존재하는 유전자만 필터
all_var = set(adata.raw.var_names)
metab_sets_filtered = {}
for nm, gl in metab_sets.items():
    kept    = [g for g in gl if g in all_var]
    missing = [g for g in gl if g not in all_var]
    metab_sets_filtered[nm] = kept
    print(f"{nm:20s}: {len(kept):3d}/{len(gl)} genes retained"
          + (f"  | missing: {missing}" if missing else ''))

metab_gs_names = list(metab_sets_filtered.keys())
print(f"\nGene sets ready: {metab_gs_names}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6b. Gene Set Score (sc.tl.score_genes) — UMAP + Violin + Dotplot
# ══════════════════════════════════════════════════════════════════════════

for nm, gl in metab_sets_filtered.items():
    if len(gl) < 3:
        print(f"Skipping {nm}: too few genes ({len(gl)})")
        continue
    sc.tl.score_genes(adata, gene_list=gl, score_name=f"{nm}_score")
    print(f"{nm}_score computed.")

metab_score_cols = [f"{nm}_score" for nm in metab_gs_names
                    if f"{nm}_score" in adata.obs.columns]

# ── UMAP ─────────────────────────────────────────────────────────────────
sc.pl.umap(adata, color=metab_score_cols, cmap='RdBu_r', vcenter=0,
           frameon=False, size=10, ncols=2, show=False)
plt.savefig(f"{FIG_DIR}/metab_scores_umap.pdf", bbox_inches='tight', dpi=300)
plt.show()

# ── Violin ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(metab_score_cols), 1,
                          figsize=(10, 3 * len(metab_score_cols)))
if len(metab_score_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, metab_score_cols):
    sc.pl.violin(adata, keys=col, groupby='celltype',
                 ax=ax, show=False, stripplot=False, jitter=False)
    ax.set_title(col.replace('_score', ''), fontsize=11, fontweight='bold')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/metab_scores_violin.pdf", bbox_inches='tight', dpi=300)
plt.show()

# ── Dotplot ───────────────────────────────────────────────────────────────
sc.pl.dotplot(adata, var_names=metab_score_cols, groupby='celltype',
              standard_scale='var', show=False)
plt.savefig(f"{FIG_DIR}/metab_scores_dotplot.pdf", bbox_inches='tight', dpi=300)
plt.show()
print("Score plots saved.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6c. Wilcoxon DEG — 전체 클러스터 (each vs rest), pts=True
# ══════════════════════════════════════════════════════════════════════════

sc.tl.rank_genes_groups(
    adata,
    groupby='celltype',
    method='wilcoxon',
    use_raw=True,
    pts=True,
    key_added='rank_allclusters_vsRest',
)
print("DEG (all clusters vs rest) complete.")

deg_all = sc.get.rank_genes_groups_df(adata, group=None,
                                       key='rank_allclusters_vsRest')
deg_all.to_csv(
    f"{FIG_DIR}/MASLD_mono_DEG_allclusters.csv", index=False
)
print(f"DEG saved. Shape: {deg_all.shape}")
deg_all.head()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6d. Preranked GSEA — 모든 클러스터 × 4 metabolic pathways
# ══════════════════════════════════════════════════════════════════════════

if not GSEAPY_AVAILABLE:
    print("gseapy not installed: pip install gseapy")
else:
    cluster_ids = sorted(adata.obs['celltype'].unique(),
                         key=lambda x: int(x) if str(x).isdigit() else x)
    cluster_ids = [str(c) for c in cluster_ids]
    print(f"Clusters to run: {cluster_ids}")

    gsea_results = {}   # {cluster_id: gseapy result}
    nes_mat  = np.full((len(cluster_ids), len(metab_gs_names)), np.nan)
    pval_mat = np.ones ((len(cluster_ids), len(metab_gs_names)))
    fdr_mat  = np.ones ((len(cluster_ids), len(metab_gs_names)))

    for ci, cl in enumerate(cluster_ids):
        print(f"  Running GSEA: Cluster {cl} vs Rest ...", end=' ')
        rnk_df = sc.get.rank_genes_groups_df(
            adata, group=cl, key='rank_allclusters_vsRest'
        )
        rnk = (rnk_df.set_index('names')['scores']
                     .dropna()
                     .sort_values(ascending=False))
        rnk = rnk[~rnk.index.duplicated(keep='first')]

        res = gp.prerank(
            rnk=rnk,
            gene_sets=metab_sets_filtered,
            min_size=3,
            max_size=5000,
            permutation_num=1000,
            outdir=None,
            seed=42,
            verbose=False,
        )
        gsea_results[cl] = res

        res_idx = res.res2d.set_index('Term')
        for gi, gs in enumerate(metab_gs_names):
            if gs not in res_idx.index:
                continue
            nes_mat[ci, gi]  = float(res_idx.loc[gs, 'NES'])
            pval_mat[ci, gi] = float(res_idx.loc[gs, 'NOM p-val'])
            fdr_col = 'FDR q-val' if 'FDR q-val' in res_idx.columns else 'fdr'
            if fdr_col in res_idx.columns:
                fdr_mat[ci, gi] = float(res_idx.loc[gs, fdr_col])
        print("done")

    # DataFrame으로 저장
    nes_df  = pd.DataFrame(nes_mat,  index=cluster_ids, columns=metab_gs_names)
    pval_df = pd.DataFrame(pval_mat, index=cluster_ids, columns=metab_gs_names)
    fdr_df  = pd.DataFrame(fdr_mat,  index=cluster_ids, columns=metab_gs_names)

    nes_df.to_csv(f"{FIG_DIR}/MASLD_mono_metab_GSEA_NES.csv")
    fdr_df.to_csv(f"{FIG_DIR}/MASLD_mono_metab_GSEA_FDR.csv")

    print("\n[NES matrix]")
    print(nes_df.round(3).to_string())
    print("\n[FDR q-val matrix]")
    print(fdr_df.round(4).to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6e. NES Heatmap — clusters × metabolic pathways
#     셀 내부: NES 수치 + NOM p-val 기반 별표
# ══════════════════════════════════════════════════════════════════════════
if GSEAPY_AVAILABLE:
    def sig_star(p):
        return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''

    vabs = np.nanmax(np.abs(nes_mat))
    fig, ax = plt.subplots(
        figsize=(len(metab_gs_names) * 2 + 1.5,
                 len(cluster_ids) * 0.7 + 1.5)
    )
    im = ax.imshow(nes_mat, cmap='RdBu_r', vmin=-vabs, vmax=vabs, aspect='auto')
    ax.set_xticks(range(len(metab_gs_names)))
    ax.set_xticklabels(metab_gs_names, fontsize=11, rotation=20, ha='right')
    ax.set_yticks(range(len(cluster_ids)))
    ax.set_yticklabels([f'Cluster {c}' for c in cluster_ids], fontsize=10)

    for ci in range(len(cluster_ids)):
        for gi in range(len(metab_gs_names)):
            nes_v  = nes_mat[ci, gi]
            pval_v = pval_mat[ci, gi]   # ← FDR → NOM p-val로 변경
            stars  = sig_star(pval_v)
            if np.isnan(nes_v):
                ax.text(gi, ci, 'NA', ha='center', va='center', fontsize=8, color='grey')
                continue
            txt_color = 'white' if abs(nes_v) > vabs * 0.6 else 'black'
            ax.text(gi, ci,
                    f"{nes_v:.2f}" + (f"\n{stars}" if stars else ''),
                    ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color=txt_color)

    cbar = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label('NES', fontsize=10)
    ax.set_title(
        'Metabolic Pathway GSEA — NES (each cluster vs Rest)\n'
        'Stars = NOM p-val (* <0.05, ** <0.01, *** <0.001)',  # ← 범례 텍스트도 변경
        fontsize=12, fontweight='bold', pad=12
    )
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/metab_GSEA_NES_heatmap.pdf", bbox_inches='tight', dpi=300)
    plt.show()
    print("NES heatmap saved.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6f. Bubble Plot — NES (color) × -log10(NOM p-val) (size)
#     유의한 결과만 선명하게 표시
# ══════════════════════════════════════════════════════════════════════════
if GSEAPY_AVAILABLE:
    rows_b = []
    for ci, cl in enumerate(cluster_ids):
        for gi, gs in enumerate(metab_gs_names):
            nes_v = nes_mat[ci, gi]
            fdr_v = fdr_mat[ci, gi]
            pv    = pval_mat[ci, gi]
            rows_b.append({
                'cluster': f'C{cl}',
                'pathway': gs,
                'NES':     nes_v,
                'FDR':     fdr_v,
                'pval':    pv,
                'neglog10_pval': -np.log10(max(pv, 1e-10)),   # ← FDR → pval
                'sig':     pv < 0.05,                          # ← FDR → pval
            })
    bubble_df = pd.DataFrame(rows_b)
    x_order = metab_gs_names
    y_order = [f'C{c}' for c in cluster_ids]
    x_idx   = {g: i for i, g in enumerate(x_order)}
    y_idx   = {c: i for i, c in enumerate(y_order)}
    vabs_b  = bubble_df['NES'].abs().max()
    size_max = 400
    fig, ax = plt.subplots(
        figsize=(len(x_order) * 2 + 1.5, len(y_order) * 0.65 + 1.8)
    )
    norm = plt.Normalize(vmin=-vabs_b, vmax=vabs_b)
    cmap = plt.cm.RdBu_r
    for _, row in bubble_df.iterrows():
        xi = x_idx[row['pathway']]
        yi = y_idx[row['cluster']]
        if np.isnan(row['NES']):
            continue
        size  = max(8, min(size_max, row['neglog10_pval'] * 60))  # ← 변경
        alpha = 0.85 if row['sig'] else 0.25
        ec    = 'black' if row['sig'] else 'grey'
        lw    = 0.8    if row['sig'] else 0.3
        ax.scatter(xi, yi, s=size, c=[cmap(norm(row['NES']))],
                   alpha=alpha, edgecolors=ec, linewidths=lw, zorder=3)
        if row['sig']:
            ax.text(xi, yi, sig_star(row['pval']),              # ← FDR → pval
                    ha='center', va='center', fontsize=7, fontweight='bold',
                    color='white' if abs(row['NES']) > vabs_b * 0.5 else 'black')
    ax.set_xticks(range(len(x_order)))
    ax.set_xticklabels(x_order, fontsize=11, rotation=20, ha='right')
    ax.set_yticks(range(len(y_order)))
    ax.set_yticklabels(y_order, fontsize=10)
    ax.set_xlim(-0.6, len(x_order) - 0.4)
    ax.set_ylim(-0.6, len(y_order) - 0.4)
    ax.grid(True, lw=0.3, alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
    cbar.set_label('NES', fontsize=10)
    import matplotlib.lines as mlines
    size_handles = [
        mlines.Line2D([0],[0], marker='o', color='w',
                      markerfacecolor='grey', markersize=np.sqrt(v * 60),
                      label=f'-log10(NOM p)={v}')                # ← 텍스트 변경
        for v in [1, 2, 3]
    ]
    ax.legend(handles=size_handles, title='Bubble size', loc='lower right',
              fontsize=8, title_fontsize=8, framealpha=0.4)
    ax.set_title(
        'Metabolic Pathway GSEA — Bubble Plot\n'
        'Color=NES  Size=-log10(NOM p-val)  Opaque=NOM p<0.05',  # ← 변경
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/metab_GSEA_bubble.pdf", bbox_inches='tight', dpi=300)
    plt.show()
    print("Bubble plot saved.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6g. Enrichment Score Curves — 각 pathway별 모든 cluster overlay
#     NES top3 / bottom3 cluster를 선명하게, 나머지는 연하게 표시
# ══════════════════════════════════════════════════════════════════════════
if GSEAPY_AVAILABLE:
    import matplotlib.cm as cm
    def fmt_pval(p):
        if np.isnan(p): return 'NA'
        return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'
    n_pw = len(metab_gs_names)
    fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5),
                              constrained_layout=True)
    if n_pw == 1:
        axes = [axes]
    cmap_cl = cm.get_cmap('tab10', len(cluster_ids))
    for ax, gs in zip(axes, metab_gs_names):
        gi = metab_gs_names.index(gs)
        nes_col   = nes_mat[:, gi]
        valid_idx = [i for i in range(len(cluster_ids)) if not np.isnan(nes_col[i])]
        sorted_by_nes = sorted(valid_idx, key=lambda i: nes_col[i], reverse=True)
        top_n    = set(sorted_by_nes[:3])
        bottom_n = set(sorted_by_nes[-3:])
        highlight = top_n | bottom_n
        legend_handles = []
        for ci, cl in enumerate(cluster_ids):
            r = gsea_results[cl].results.get(gs, {})
            es_curve = r.get('RES', None)
            if es_curve is None or len(es_curve) == 0:
                continue
            nes_v  = nes_mat[ci, gi]
            pval_v = pval_mat[ci, gi]                           # ← FDR → pval
            x_rank = np.linspace(0, 1, len(es_curve))
            color  = cmap_cl(ci)
            if ci in highlight:
                lw    = 2.2
                alpha = 0.9
                lbl   = (f"C{cl}  NES={nes_v:.2f}  "
                         f"NOM p={fmt_pval(pval_v)}"           # ← FDR= → NOM p=
                         f"{' ' + sig_star(pval_v) if sig_star(pval_v) else ''}")  # ← 변경
                legend_handles.append(
                    plt.Line2D([0],[0], color=color, lw=2, label=lbl)
                )
            else:
                lw    = 0.8
                alpha = 0.25
            ax.plot(x_rank, es_curve, color=color, lw=lw, alpha=alpha)
        ax.axhline(0, color='grey', lw=0.8, ls='--')
        ax.set_title(gs, fontsize=12, fontweight='bold')
        ax.set_xlabel('Gene rank (normalized)', fontsize=9)
        ax.set_ylabel('Running ES', fontsize=9)
        ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,
                  loc='best', title='Top/Bottom 3 by NES', title_fontsize=7)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    fig.suptitle(
        'GSEA Enrichment Score Curves — Metabolic Pathways (all clusters)\n'
        '(Bold = NES top/bottom 3 per pathway)',
        fontsize=12, fontweight='bold'
    )
    plt.savefig(f"{FIG_DIR}/metab_GSEA_ES_curves.pdf", bbox_inches='tight', dpi=300)
    plt.show()
    print("ES curve plot saved.")

In [ ]:
highlight_clusters = ["Classical_monocyte", "SLC7A5_high_monocyte"]

highlight_idx = {
    i for i, cl in enumerate(cluster_ids)
    if cl in highlight_clusters
}

if GSEAPY_AVAILABLE:
    import matplotlib.cm as cm

    def fmt_pval(p):
        if np.isnan(p): return 'NA'
        return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

    n_pw = len(metab_gs_names)
    fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5),
                              constrained_layout=True)
    if n_pw == 1:
        axes = [axes]

    cmap_cl = cm.get_cmap('tab10', len(cluster_ids))

    for ax, gs in zip(axes, metab_gs_names):
        gi = metab_gs_names.index(gs)

        legend_handles = []
        for ci, cl in enumerate(cluster_ids):
            r = gsea_results[cl].results.get(gs, {})
            es_curve = r.get('RES', None)
            if es_curve is None or len(es_curve) == 0:
                continue
            nes_v  = nes_mat[ci, gi]
            pval_v = pval_mat[ci, gi]                           # ← FDR → pval
            x_rank = np.linspace(0, 1, len(es_curve))
            color  = cmap_cl(ci)
            if ci in highlight_idx:
                lw    = 2.2
                alpha = 0.9
                lbl   = (f"C{cl}  NES={nes_v:.2f}  "
                         f"NOM p={fmt_pval(pval_v)}"           # ← FDR= → NOM p=
                         f"{' ' + sig_star(pval_v) if sig_star(pval_v) else ''}")  # ← 변경
                legend_handles.append(
                    plt.Line2D([0],[0], color=color, lw=2, label=lbl)
                )
            else:
                lw    = 0.8
                alpha = 0.25

            ax.plot(x_rank, es_curve, color=color, lw=lw, alpha=alpha)

        ax.axhline(0, color='grey', lw=0.8, ls='--')
        ax.set_title(gs, fontsize=12, fontweight='bold')
        ax.set_xlabel('Gene rank (normalized)', fontsize=9)
        ax.set_ylabel('Running ES', fontsize=9)
        ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,
                  loc='best', title='Top/Bottom 3 by NES', title_fontsize=7)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig.suptitle(
        'GSEA Enrichment Score Curves — Metabolic Pathways (all clusters)\n'
        '(Bold = NES top/bottom 3 per pathway)',
        fontsize=12, fontweight='bold'
    )
    plt.savefig(f"{FIG_DIR}/monocyte_GSEA_ES_curves.pdf", bbox_inches='tight', dpi=300)
    plt.show()
    print("ES curve plot saved.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6h. NES Bar Chart — 각 pathway별 cluster NES (NOM p-val 기준 색상 강조)
# ══════════════════════════════════════════════════════════════════════════
if GSEAPY_AVAILABLE:
    n_pw = len(metab_gs_names)
    fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 4, 5),
                              sharey=False, constrained_layout=True)
    if n_pw == 1:
        axes = [axes]
    for ax, gs in zip(axes, metab_gs_names):
        gi     = metab_gs_names.index(gs)
        nes_v  = nes_mat[:, gi]
        pval_v = pval_mat[:, gi]                               # ← [ci,gi] → [:,gi] 핵심 수정
        y_pos  = np.arange(len(cluster_ids))
        colors = [
            '#d62728' if n > 0 and f < 0.05 else
            '#1f77b4' if n < 0 and f < 0.05 else
            '#d62728' if n > 0 else '#1f77b4'
            for n, f in zip(nes_v, pval_v)
        ]
        alphas = [0.9 if f < 0.05 else 0.35 for f in pval_v]
        for yi, (n, c, a, f) in enumerate(zip(nes_v, colors, alphas, pval_v)):
            if np.isnan(n):
                continue
            ax.barh(yi, n, color=c, alpha=a, height=0.7)
            stars = sig_star(f)
            if stars:
                ax.text(n + (0.03 if n >= 0 else -0.03), yi,
                        stars, va='center',
                        ha='left' if n >= 0 else 'right',
                        fontsize=9, fontweight='bold',
                        color='black')
        ax.axvline(0, color='black', lw=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels([f'C{c}' for c in cluster_ids], fontsize=9)
        ax.set_xlabel('NES (vs Rest)', fontsize=9)
        ax.set_title(gs, fontsize=11, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    import matplotlib.patches as mpatches
    fig.legend(handles=[
        mpatches.Patch(color='#d62728', alpha=0.9, label='NES>0, NOM p<0.05'),    # ← 텍스트 변경
        mpatches.Patch(color='#1f77b4', alpha=0.9, label='NES<0, NOM p<0.05'),    # ← 텍스트 변경
        mpatches.Patch(color='grey',    alpha=0.35, label='NOM p≥0.05'),           # ← 텍스트 변경
    ], loc='lower center', ncol=3, fontsize=9, frameon=True,
               bbox_to_anchor=(0.5, -0.06))
    fig.suptitle('GSEA NES per Pathway — All Clusters\n'
                 '(Stars = NOM p-val significance)',                                # ← 텍스트 변경
                 fontsize=12, fontweight='bold')
    plt.savefig(f"{FIG_DIR}/metab_GSEA_NES_barplot.pdf", bbox_inches='tight', dpi=300)
    plt.show()
    print("Bar chart saved.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Classical_monocyte vs SLC7A5_high_monocyte — Preranked GSEA
# ══════════════════════════════════════════════════════════════════════════

group_A = 'Classical_monocyte'
group_B = 'SLC7A5_high_monocyte'

# ── 1. 두 그룹만 subset ──────────────────────────────────────────────────
mask = adata.obs['celltype'].isin([group_A, group_B])
adata_sub = adata[mask].copy()

# ── 2. A vs B로 rank_genes_groups 새로 계산 ──────────────────────────────
sc.tl.rank_genes_groups(
    adata_sub,
    groupby='celltype',
    groups=[group_A],
    reference=group_B,
    method='wilcoxon',
    key_added='rank_AvsB',
)

# ── 3. Ranked list 추출 ───────────────────────────────────────────────────
rnk_df = sc.get.rank_genes_groups_df(
    adata_sub, group=group_A, key='rank_AvsB'
)
rnk = (rnk_df.set_index('names')['scores']
             .dropna()
             .sort_values(ascending=False))
rnk = rnk[~rnk.index.duplicated(keep='first')]

print(f"Ranked genes: {len(rnk)}")
print(f"Top 5:    {rnk.head().to_dict()}")
print(f"Bottom 5: {rnk.tail().to_dict()}")

# ── 4. Preranked GSEA ─────────────────────────────────────────────────────
res = gp.prerank(
    rnk=rnk,
    gene_sets=metab_sets_filtered,
    min_size=3,
    max_size=5000,
    permutation_num=1000,
    outdir=None,
    seed=42,
    verbose=False,
)

# ── 5. 결과 정리 ──────────────────────────────────────────────────────────
res_df = res.res2d.set_index('Term')
print("\n[GSEA Result: Classical vs SLC7A5_high]")
print(res_df[['NES','NOM p-val','FDR q-val','Tag %']].round(4).to_string())

# ── 6. 시각화 ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

pathways = metab_gs_names

nes_vals  = [float(res_df.loc[p, 'NES'])        if p in res_df.index else np.nan for p in pathways]
pval_vals = [float(res_df.loc[p, 'NOM p-val'])  if p in res_df.index else 1.0    for p in pathways]

# ── 색상: NOM p-val 기준 ──────────────────────────────────────────────────
colors = []
for nes, pval in zip(nes_vals, pval_vals):                     # ← fdr → pval
    if pval < 0.05:
        colors.append('#d62728' if nes > 0 else '#1f77b4')     # 진한색 = 유의
    elif pval < 0.25:
        colors.append('#f28e8e' if nes > 0 else '#90badb')     # 연한색 = 경계
    else:
        colors.append('#cccccc')                               # 회색 = 비유의

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(pathways, nes_vals, color=colors, edgecolor='black', linewidth=0.7)

# ── 유의성 별표: NOM p-val 기준 ───────────────────────────────────────────
for i, (nes, pval) in enumerate(zip(nes_vals, pval_vals)):     # ← fdr 제거
    if np.isnan(nes):
        continue
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else \
          '†' if pval < 0.25 else 'ns'                         # ← fdr → pval
    x_pos = nes + (0.05 if nes >= 0 else -0.05)
    ha = 'left' if nes >= 0 else 'right'
    ax.text(x_pos, i, sig, va='center', ha=ha, fontsize=11)

ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('NES (Normalized Enrichment Score)', fontsize=11)
ax.set_title(f'Metabolic GSEA\n{group_A}  vs  {group_B}', fontsize=12)

# ── 범례 ──────────────────────────────────────────────────────────────────
patches = [
    mpatches.Patch(color='#d62728', label=f'Higher in {group_A} (NOM p<0.05)'),   # ← 변경
    mpatches.Patch(color='#1f77b4', label=f'Higher in {group_B} (NOM p<0.05)'),   # ← 변경
    mpatches.Patch(color='#f28e8e', label='NOM p 0.05–0.25'),                      # ← 변경
    mpatches.Patch(color='#cccccc', label='ns'),
]
ax.legend(handles=patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/GSEA_Classical_vs_SLC7A5high_metabolic.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(metab_sets_filtered.keys())

for gs_name, gs_genes in metab_sets_filtered.items():
    overlap = [g for g in gs_genes if g in rnk.index]
    print(f"{gs_name}: {len(overlap)} / {len(gs_genes)}")

print(res.res2d[['Term','NES','NOM p-val','FDR q-val']].to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6i. Summary — 유의한 결과 (FDR < 0.05) 요약
# ══════════════════════════════════════════════════════════════════════════

if GSEAPY_AVAILABLE:
    print("[NES Matrix]")
    print(nes_df.round(3).to_string())
    print("\n[FDR q-val Matrix]")
    print(fdr_df.round(4).to_string())

    sig_rows = []
    for ci, cl in enumerate(cluster_ids):
        for gi, gs in enumerate(metab_gs_names):
            nes_v = nes_mat[ci, gi]
            fdr_v = fdr_mat[ci, gi]
            pv    = pval_mat[ci, gi]
            if not np.isnan(nes_v) and fdr_v < 0.05:
                sig_rows.append({
                    'cluster':  cl,
                    'pathway':  gs,
                    'NES':      round(nes_v, 3),
                    'NOM_pval': round(pv, 4),
                    'FDR_qval': round(fdr_v, 4),
                    'direction': 'UP' if nes_v > 0 else 'DOWN',
                })

    sig_df = pd.DataFrame(sig_rows).sort_values(
        ['pathway', 'NES'], ascending=[True, False]
    ).reset_index(drop=True)

    print(f"\n[Significant results, FDR < 0.05] — {len(sig_df)} entries")
    print(sig_df.to_string(index=False))

    sig_df.to_csv(f"{FIG_DIR}/MASLD_mono_metab_GSEA_significant.csv", index=False)
    print("\nSummary CSV saved.")


# GSEApy database

In [ ]:
import gseapy as gp
gp.get_library_name() 

In [ ]:
# KEGG — 가장 직관적, glycolysis/TCA/OXPHOS 다 있음
kegg_sets = gp.get_library('KEGG_2021_Human')

# Reactome — 더 세분화된 pathway
reactome_sets = gp.get_library('Reactome_2022')

# WikiPathways
wiki_sets = gp.get_library('WikiPathway_2023_Human')

In [ ]:
# ── KEGG로 대사 pathway만 필터링해서 GSEA ──────────────────────────────
kegg_sets = gp.get_library('KEGG_2021_Human')

# 관심 pathway 키워드 필터링
keywords = ['glycolysis', 'gluconeogenesis', 'citrate', 'oxidative phosphorylation',
            'fatty acid degradation', 'fatty acid oxidation']

kegg_metab = {
    k: v for k, v in kegg_sets.items()
    if any(kw.lower() in k.lower() for kw in keywords)
}

# 어떤 pathway가 걸렸는지 확인
for k in kegg_metab.keys():
    print(k)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Classical_monocyte vs SLC7A5_high_monocyte — Preranked GSEA
# ══════════════════════════════════════════════════════════════════════════

group_A = 'Classical_monocyte'
group_B = 'SLC7A5_high_monocyte'

# ── 1. 두 그룹만 subset ──────────────────────────────────────────────────
mask = adata.obs['celltype'].isin([group_A, group_B])
adata_sub = adata[mask].copy()

# ── 2. A vs B로 rank_genes_groups 새로 계산 ──────────────────────────────
sc.tl.rank_genes_groups(
    adata_sub,
    groupby='celltype',
    groups=[group_A],
    reference=group_B,
    method='wilcoxon',
    key_added='rank_AvsB',
)

# ── 3. Ranked list 추출 ───────────────────────────────────────────────────
rnk_df = sc.get.rank_genes_groups_df(
    adata_sub, group=group_A, key='rank_AvsB'
)
rnk = (rnk_df.set_index('names')['scores']
             .dropna()
             .sort_values(ascending=False))
rnk = rnk[~rnk.index.duplicated(keep='first')]

print(f"Ranked genes: {len(rnk)}")
print(f"Top 5:    {rnk.head().to_dict()}")
print(f"Bottom 5: {rnk.tail().to_dict()}")

# ── KEGG gene set 불러오기 및 필터링 ──────────────────────────────────────
kegg_sets = gp.get_library('KEGG_2021_Human')

keywords = ['glycolysis', 'gluconeogenesis', 'citrate', 'oxidative phosphorylation',
            'fatty acid']
kegg_metab = {
    k: v for k, v in kegg_sets.items()
    if any(kw.lower() in k.lower() for kw in keywords)
}

print("사용할 KEGG pathways:")
for k in kegg_metab.keys():
    print(f"  {k}")

# ── GSEA 실행 ─────────────────────────────────────────────────────────────
res_kegg = gp.prerank(
    rnk=rnk,
    gene_sets=kegg_metab,
    min_size=3,
    max_size=5000,
    permutation_num=1000,
    outdir=None,
    seed=42,
    verbose=False,
)

res_kegg_df = res_kegg.res2d.set_index('Term')
print(res_kegg_df[['NES','NOM p-val','FDR q-val','Tag %']].round(4).to_string())

# ── 시각화 ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

pathways = list(res_kegg_df.index)          # ← KEGG pathway 이름 그대로 사용

nes_vals  = [float(res_kegg_df.loc[p, 'NES'])       for p in pathways]
pval_vals = [float(res_kegg_df.loc[p, 'NOM p-val']) for p in pathways]  # ← res_kegg_df로 통일

colors = []
for nes, pval in zip(nes_vals, pval_vals):
    if pval < 0.05:
        colors.append('#d62728' if nes > 0 else '#1f77b4')
    elif pval < 0.25:
        colors.append('#f28e8e' if nes > 0 else '#90badb')
    else:
        colors.append('#cccccc')

fig, ax = plt.subplots(figsize=(8, len(pathways) * 0.6 + 1.5))  # ← pathway 수에 맞게 자동 조절
bars = ax.barh(pathways, nes_vals, color=colors, edgecolor='black', linewidth=0.7)

for i, (nes, pval) in enumerate(zip(nes_vals, pval_vals)):
    if np.isnan(nes):
        continue
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else \
          '†' if pval < 0.25 else 'ns'
    x_pos = nes + (0.05 if nes >= 0 else -0.05)
    ha = 'left' if nes >= 0 else 'right'
    ax.text(x_pos, i, sig, va='center', ha=ha, fontsize=11)

ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('NES (Normalized Enrichment Score)', fontsize=11)
ax.set_title(f'Metabolic GSEA (KEGG)\n{group_A}  vs  {group_B}', fontsize=12)

patches = [
    mpatches.Patch(color='#d62728', label=f'Higher in {group_A} (NOM p<0.05)'),
    mpatches.Patch(color='#1f77b4', label=f'Higher in {group_B} (NOM p<0.05)'),
    mpatches.Patch(color='#f28e8e', label='NOM p 0.05–0.25'),
    mpatches.Patch(color='#cccccc', label='ns'),
]
ax.legend(handles=patches, fontsize=8, loc='lower right')

plt.tight_layout()
#plt.savefig(f"{FIG_DIR}/GSEA_Classical_vs_SLC7A5high_KEGG.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(gsea_results.keys())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ES Curves — Classical & SLC7A5_high 두 클러스터 overlay
#             (gsea_results = vs Rest 결과 사용)
# ══════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import numpy as np

highlight_clusters = ["Classical_monocyte", "SLC7A5_high_monocyte"]
colors_map = {
    "Classical_monocyte":    '#d62728',   # 빨강
    "SLC7A5_high_monocyte":  '#1f77b4',   # 파랑
}

def fmt_pval(p):
    if np.isnan(p): return 'NA'
    return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

n_pw = len(metab_gs_names)
fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5),
                          constrained_layout=True)
if n_pw == 1:
    axes = [axes]

for ax, gs in zip(axes, metab_gs_names):
    gi = metab_gs_names.index(gs)
    legend_handles = []

    for cl in highlight_clusters:
        ci = cluster_ids.index(cl)

        r = gsea_results[cl].results.get(gs, {})
        es_curve = r.get('RES', None)
        if es_curve is None or len(es_curve) == 0:
            continue

        nes_v  = nes_mat[ci, gi]
        pval_v = pval_mat[ci, gi]
        x_rank = np.linspace(0, 1, len(es_curve))
        color  = colors_map[cl]

        ax.plot(x_rank, es_curve, color=color, lw=2.2, alpha=0.9)

        # 최고점 표시
        peak_idx = np.argmax(np.abs(es_curve))
        ax.scatter(x_rank[peak_idx], es_curve[peak_idx],
                   color=color, s=40, zorder=5)

        stars = sig_star(pval_v)
        lbl = (f"{cl}\nNES={nes_v:.2f}  NOM p={fmt_pval(pval_v)}"
               f"{' ' + stars if stars else ''}")
        legend_handles.append(
            plt.Line2D([0],[0], color=color, lw=2, label=lbl)
        )

    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.set_title(gs, fontsize=12, fontweight='bold')
    ax.set_xlabel('Gene rank (normalized)', fontsize=9)
    ax.set_ylabel('Running ES', fontsize=9)
    ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,
              loc='best', title_fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    f'GSEA Enrichment Score Curves\nClassical_monocyte  vs  SLC7A5_high_monocyte (each vs Rest)',
    fontsize=12, fontweight='bold'
)
plt.savefig(f"{FIG_DIR}/GSEA_highlight_ES_curves.pdf", bbox_inches='tight', dpi=300)
plt.show()
print("ES curve plot saved.")

In [ ]:
# ── KEGG로 대사 pathway만 필터링해서 GSEA ──────────────────────────────
kegg_sets = gp.get_library('KEGG_2021_Human')

# 관심 pathway 키워드 필터링
keywords = ['mtor', 'pi3k', 'akt', 'hif',
            'glycolysis']

kegg_metab = {
    k: v for k, v in kegg_sets.items()
    if any(kw.lower() in k.lower() for kw in keywords)
}

# 어떤 pathway가 걸렸는지 확인
for k in kegg_metab.keys():
    print(k)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Classical_monocyte vs SLC7A5_high_monocyte — Preranked GSEA
# ══════════════════════════════════════════════════════════════════════════

group_A = 'Classical_monocyte'
group_B = 'SLC7A5_high_monocyte'

# ── 1. 두 그룹만 subset ──────────────────────────────────────────────────
mask = adata.obs['celltype'].isin([group_A, group_B])
adata_sub = adata[mask].copy()

# ── 2. A vs B로 rank_genes_groups 새로 계산 ──────────────────────────────
sc.tl.rank_genes_groups(
    adata_sub,
    groupby='celltype',
    groups=[group_A],
    reference=group_B,
    method='wilcoxon',
    key_added='rank_AvsB',
)

# ── 3. Ranked list 추출 ───────────────────────────────────────────────────
rnk_df = sc.get.rank_genes_groups_df(
    adata_sub, group=group_A, key='rank_AvsB'
)
rnk = (rnk_df.set_index('names')['scores']
             .dropna()
             .sort_values(ascending=False))
rnk = rnk[~rnk.index.duplicated(keep='first')]

print(f"Ranked genes: {len(rnk)}")
print(f"Top 5:    {rnk.head().to_dict()}")
print(f"Bottom 5: {rnk.tail().to_dict()}")

# ── KEGG gene set 불러오기 및 필터링 ──────────────────────────────────────
kegg_sets = gp.get_library('KEGG_2021_Human')

keywords = ['mtor', 'pi3k', 'akt', 'hif','glycolysis', 'gluconeogenesis', 'citrate', 'oxidative phosphorylation',
            'fatty acid']
kegg_metab = {
    k: v for k, v in kegg_sets.items()
    if any(kw.lower() in k.lower() for kw in keywords)
}

print("사용할 KEGG pathways:")
for k in kegg_metab.keys():
    print(f"  {k}")

# ── GSEA 실행 ─────────────────────────────────────────────────────────────
res_kegg = gp.prerank(
    rnk=rnk,
    gene_sets=kegg_metab,
    min_size=3,
    max_size=5000,
    permutation_num=1000,
    outdir=None,
    seed=42,
    verbose=False,
)

res_kegg_df = res_kegg.res2d.set_index('Term')
print(res_kegg_df[['NES','NOM p-val','FDR q-val','Tag %']].round(4).to_string())

# ── 시각화 ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

pathways = list(res_kegg_df.index)          # ← KEGG pathway 이름 그대로 사용

nes_vals  = [float(res_kegg_df.loc[p, 'NES'])       for p in pathways]
pval_vals = [float(res_kegg_df.loc[p, 'NOM p-val']) for p in pathways]  # ← res_kegg_df로 통일

colors = []
for nes, pval in zip(nes_vals, pval_vals):
    if pval < 0.05:
        colors.append('#d62728' if nes > 0 else '#1f77b4')
    elif pval < 0.25:
        colors.append('#f28e8e' if nes > 0 else '#90badb')
    else:
        colors.append('#cccccc')

fig, ax = plt.subplots(figsize=(8, len(pathways) * 0.6 + 1.5))  # ← pathway 수에 맞게 자동 조절
bars = ax.barh(pathways, nes_vals, color=colors, edgecolor='black', linewidth=0.7)

for i, (nes, pval) in enumerate(zip(nes_vals, pval_vals)):
    if np.isnan(nes):
        continue
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else \
          '†' if pval < 0.25 else 'ns'
    x_pos = nes + (0.05 if nes >= 0 else -0.05)
    ha = 'left' if nes >= 0 else 'right'
    ax.text(x_pos, i, sig, va='center', ha=ha, fontsize=11)

ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('NES (Normalized Enrichment Score)', fontsize=11)
ax.set_title(f'Metabolic GSEA (KEGG)\n{group_A}  vs  {group_B}', fontsize=12)

patches = [
    mpatches.Patch(color='#d62728', label=f'Higher in {group_A} (NOM p<0.05)'),
    mpatches.Patch(color='#1f77b4', label=f'Higher in {group_B} (NOM p<0.05)'),
    mpatches.Patch(color='#f28e8e', label='NOM p 0.05–0.25'),
    mpatches.Patch(color='#cccccc', label='ns'),
]
ax.legend(handles=patches, fontsize=8, loc='lower right')

plt.tight_layout()
#plt.savefig(f"{FIG_DIR}/GSEA_Classical_vs_SLC7A5high_KEGG.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── GSEA curve (A vs B prerank 결과 기반) ────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# 🔹 pathway 선택 (bar plot과 동일하게)
selected_pathways = pathways   # or pathways[:5] 등으로 줄여도 됨

n_pw = len(selected_pathways)

fig, axes = plt.subplots(
    1, n_pw,
    figsize=(5 * n_pw, 4),
    constrained_layout=True
)

if n_pw == 1:
    axes = [axes]

for ax, pw in zip(axes, selected_pathways):

    r = res_kegg.results.get(pw, None)
    if r is None:
        continue

    es_curve = r['RES']
    hits = r['hits']

    x = np.linspace(0, 1, len(es_curve))

    # 🔹 NES 기반 색상 (bar plot과 동일한 컨셉 유지)
    nes = res_kegg_df.loc[pw, 'NES']
    pval = res_kegg_df.loc[pw, 'NOM p-val']

    if pval < 0.05:
        color = '#d62728' if nes > 0 else '#1f77b4'
    elif pval < 0.25:
        color = '#f28e8e' if nes > 0 else '#90badb'
    else:
        color = '#888888'

    # ── ES curve ─────────────────────────────────────
    ax.plot(x, es_curve, color=color, lw=2)

    # ── hit ticks ────────────────────────────────────
    ax.vlines(
        np.array(hits) / len(es_curve),
        ymin=min(es_curve),
        ymax=min(es_curve) + 0.1,
        color='black',
        linewidth=0.6,
        alpha=0.6
    )

    # ── baseline ─────────────────────────────────────
    ax.axhline(0, color='grey', linestyle='--', lw=1)

    # ── peak 표시 ────────────────────────────────────
    peak_idx = np.argmax(np.abs(es_curve))
    ax.scatter(
        x[peak_idx],
        es_curve[peak_idx],
        color=color,
        s=40,
        zorder=5
    )

    # ── title ────────────────────────────────────────
    ax.set_title(
        f"{pw}\nNES={nes:.2f} | p={pval:.3f}",
        fontsize=10
    )

    ax.set_xlabel("Gene rank (normalized)")
    ax.set_ylabel("Running ES")

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# ── 전체 제목 ─────────────────────────────────────────
fig.suptitle(
    f"GSEA Enrichment Curves (Preranked)\n{group_A} vs {group_B}",
    fontsize=13,
    fontweight='bold'
)

plt.show()

---
## 7. Analysis Complete

### 저장된 Figure / Data 목록
| File | Content |
|------|---------|
| `metab_scores_umap.pdf` | 4개 metabolic score UMAP |
| `metab_scores_violin.pdf` | cluster별 score violin |
| `metab_scores_dotplot.pdf` | cluster별 score dotplot |
| `MASLD_mono_DEG_allclusters.csv` | Wilcoxon DEG (all clusters vs rest) |
| `MASLD_mono_metab_GSEA_NES.csv` | NES matrix (cluster × pathway) |
| `MASLD_mono_metab_GSEA_FDR.csv` | FDR matrix |
| `metab_GSEA_NES_heatmap.pdf` | NES heatmap |
| `metab_GSEA_bubble.pdf` | Bubble plot (NES × FDR) |
| `metab_GSEA_ES_curves.pdf` | ES curve overlay |
| `metab_GSEA_NES_barplot.pdf` | NES bar chart per pathway |
| `MASLD_mono_metab_GSEA_significant.csv` | FDR<0.05 유의 결과 |
